In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [ ]:
def load_selected(path):
    df = pd.read_csv(path)
    y_class = df['Y_Class'].copy()
    drop_cols = ['Y_Quality', 'Y_Class', 'PRODUCT_ID', 'PRODUCT_CODE', 'TIMESTAMP']
    X = df.drop(columns=drop_cols)
    return X, y_class

In [ ]:
def get_models(random_state=42):
    return {
        'LightGBM': LGBMClassifier(random_state=random_state, verbose=-1),
        'XGBoost': XGBClassifier(random_state=random_state, eval_metric='mlogloss'),
        'CatBoost': CatBoostClassifier(random_state=random_state, verbose=0),
        'RandomForest': RandomForestClassifier(random_state=random_state),
        'HistGradientBoosting': HistGradientBoostingClassifier(random_state=random_state),
    }

In [ ]:
def evaluate_cv_multi(X, y_class, n_splits=5, random_state=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    names = get_models(random_state).keys()
    model_results = {name: {'fold_f1': [], 'true': [], 'pred': []} for name in names}

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_class)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y_class.iloc[train_idx], y_class.iloc[val_idx]
        sample_weight = compute_sample_weight('balanced', y_train)

        for name, model in get_models(random_state).items():
            model.fit(X_train, y_train, sample_weight=sample_weight)
            pred = model.predict(X_val)

            f1 = f1_score(y_val, pred, average='macro')
            model_results[name]['fold_f1'].append(f1)
            model_results[name]['true'].extend(y_val.tolist())
            model_results[name]['pred'].extend(pred.tolist())

    for name, res in model_results.items():
        print(f'--- {name} ---')
        print(f'fold 평균 macro-F1: {np.mean(res["fold_f1"]):.4f} (std {np.std(res["fold_f1"]):.4f})')
        print(classification_report(res['true'], res['pred'], labels=[0, 1, 2], zero_division=0))

    return model_results

In [ ]:
print('===== A_31 / v2 (27 features) =====')
X_A_v2, y_A_v2 = load_selected('A_31_selected_v2.csv')
results_A_v2 = evaluate_cv_multi(X_A_v2, y_A_v2)

In [ ]:
print('===== TO_31 / v1 (52 features) =====')
X_TO_v1, y_TO_v1 = load_selected('TO_31_selected.csv')
results_TO_v1 = evaluate_cv_multi(X_TO_v1, y_TO_v1)

In [ ]:
print('===== TO_31 / v2 (8 features) =====')
X_TO_v2, y_TO_v2 = load_selected('TO_31_selected_v2.csv')
results_TO_v2 = evaluate_cv_multi(X_TO_v2, y_TO_v2)

In [ ]:
rows = []
for combo_name, results in [('A_31 v2', results_A_v2), ('TO_31 v1', results_TO_v1), ('TO_31 v2', results_TO_v2)]:
    for model_name, res in results.items():
        rows.append({
            '조합': combo_name,
            '모델': model_name,
            'fold 평균 macro-F1': np.mean(res['fold_f1']),
            'fold std': np.std(res['fold_f1'])
        })
summary = pd.DataFrame(rows)
summary.sort_values(['조합', 'fold 평균 macro-F1'], ascending=[True, False])